In [0]:
# SCD Type 1: upsert source into target Delta table (latest record wins, no history kept)
from delta.tables import DeltaTable
from pyspark.sql import functions as F

source_table = "main.default.customers_source"   # incoming data (e.g. latest daily extract)
target_table = "main.default.customers_target"     # SCD1 Delta table
join_keys = ["customer_id"]                        # business / natural key(s)

source_df = spark.table(source_table).dropDuplicates(subset=join_keys)  # keep one row per key

# Column comparison: update only when any tracked attribute actually changed
tracked_cols = [c for c in source_df.columns if c not in join_keys]
update_condition = " OR ".join(
    [f"target.{c} IS DISTINCT FROM source.{c}" for c in tracked_cols]
)

# (One-time) create the target table if it doesn't exist yet
if not spark.catalog.tableExists(target_table):
    source_df.write.format("delta").saveAsTable(target_table)
else:
    target_dt = DeltaTable.forName(spark, target_table)

    target_dt.alias("target").merge(
        source_df.alias("source"),
        " AND ".join([f"target.{k} = source.{k}" for k in join_keys]),
    ).whenMatchedUpdate(
        condition=update_condition,
        set={c: f"source.{c}" for c in tracked_cols},
    ).whenNotMatchedInsert(
        values={c: f"source.{c}" for c in source_df.columns},
    ).execute()

print(f"SCD1 merge completed into {target_table}")
spark.table(target_table).display()